In [70]:
import pandas as pd
import numpy as np
import sys
from scipy.fft import rfft, rfftfreq

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [45]:
df = pd.read_csv('data/df_processed.csv')
df = df[["time", "subject", "exercise", "set_nr", "focus", "hr", *[col for col in df.columns if col.endswith("lowpass")]]]
df["set_id"] = (
    df.groupby(["subject", "exercise", "set_nr", "focus"]).ngroup()
)

In [27]:
df.head()

,time,subject,exercise,set_nr,focus,hr,acc_lin_x_lowpass,acc_lin_y_lowpass,acc_lin_z_lowpass,acc_x_lowpass,acc_y_lowpass,acc_z_lowpass,gyro_x_lowpass,gyro_y_lowpass,gyro_z_lowpass,yaw_lowpass,pitch_lowpass,roll_lowpass,set_id
0,0.00,0,0,1,1,102.00,-1.268185,0.420601,-1.101016,-7.962453,6.182413,1.292638,0.829020,-0.728038,-1.796314,67.084071,47.290973,-32.326328,0
1,0.02,0,0,1,1,102.02,-0.697262,0.747175,-0.806400,-7.711188,6.472090,1.483989,0.862181,-0.634146,-1.804213,66.738141,48.892341,-34.134206,0
2,0.04,0,0,1,1,102.04,-0.230222,0.995949,-0.530047,-7.529258,6.677234,1.684365,0.880386,-0.558087,-1.818941,66.287539,50.509295,-36.017281,0
3,0.06,0,0,1,1,102.06,0.057452,1.108086,-0.288919,-7.470036,6.729404,1.897057,0.872640,-0.512104,-1.843712,65.624422,52.152371,-38.059840,0
4,0.08,0,0,1,1,102.08,0.137882,1.058690,-0.098308,-7.558365,6.590234,2.113445,0.835011,-0.499340,-1.876353,64.663349,53.820591,-40.330600,0


In [46]:
from scipy.fft import rfft, rfftfreq
import numpy as np

SAMPLE_RATE = 50  # 20 ms intervals

sensor_cols = [c for c in df.columns if c not in ["time", "subject", "exercise", "set_nr", "focus", "set_id"]]

def fft_features(x):
    x = np.asarray(x)

    fft_vals = np.abs(rfft(x))
    freqs = rfftfreq(len(x), d=1 / SAMPLE_RATE)

    fft_vals[0] = 0

    dominant_idx = np.argmax(fft_vals)

    dominant_freq = freqs[dominant_idx]
    dominant_power = fft_vals[dominant_idx]

    p = fft_vals / (fft_vals.sum() + 1e-12)

    spectral_entropy = -(p * np.log2(p + 1e-12)).sum()
    spectral_centroid = np.sum(freqs * fft_vals) / (np.sum(fft_vals) + 1e-12)

    return {
        "dominant_freq": dominant_freq,
        "dominant_power": dominant_power,
        "spectral_entropy": spectral_entropy,
        "spectral_centroid": spectral_centroid,
    }


fft_rows = []

for set_id, group in df.groupby("set_id"):
    row = {
        "set_id": set_id,
        "subject": group["subject"].iloc[0],
        "exercise": group["exercise"].iloc[0],
        "set_nr": group["set_nr"].iloc[0],
        "focus": group["focus"].iloc[0],
    }

    for col in sensor_cols:
        feats = fft_features(group[col])

        for feat_name, value in feats.items():
            row[f"{col}_{feat_name}"] = value

    fft_rows.append(row)

fft_df = pd.DataFrame(fft_rows)

In [47]:
AXIS_TRIPLES = {
    "acc_lin": ["acc_lin_x_lowpass", "acc_lin_y_lowpass", "acc_lin_z_lowpass"],
    "acc":     ["acc_x_lowpass", "acc_y_lowpass", "acc_z_lowpass"],
    "gyro":    ["gyro_x_lowpass", "gyro_y_lowpass", "gyro_z_lowpass"],
    "orient":  ["yaw_lowpass", "pitch_lowpass", "roll_lowpass"],
}

def add_axis_combinations(g):
    new_data = {}

    for name, (cx, cy, cz) in AXIS_TRIPLES.items():
        x, y, z = g[cx], g[cy], g[cz]

        new_data[f"{name}_magnitude"] = np.sqrt(x**2 + y**2 + z**2)
        new_data[f"{name}_sum"] = x + y + z
        new_data[f"{name}_abs_sum"] = x.abs() + y.abs() + z.abs()

    new_df = pd.DataFrame(new_data, index=g.index)
    return pd.concat([g, new_df], axis=1), list(new_data.keys())


In [48]:
df, newcols = add_axis_combinations(df)

sensor_cols = [c for c in df.columns if c not in ["time", "subject", "exercise", "set_nr", "focus", "set_id"]]

agg_df = (
    df.groupby("set_id")[sensor_cols].agg(["std", "min", "max", "median"])
)

agg_df.columns = [f"{col}_{stat}" for col, stat in agg_df.columns]

agg_df = agg_df.reset_index()

svm_df = agg_df.merge(fft_df, on="set_id", how="inner")

In [58]:
print(
    len([c for c in X.columns if c.startswith("acc_lin_")]),
    len([c for c in X.columns if c.startswith(("yaw_", "pitch_", "roll_", "orient_"))]),
    len([c for c in X.columns if c.startswith("gyro_")]),
    len([c for c in X.columns if c.startswith("acc_") and not c.startswith("acc_lin_")]),
    len([c for c in X.columns if c.startswith("hr_")]),
)

36 36 36 36 8


In [63]:

# testing how much PCA components to use
groups = {
    "acc_lin": [c for c in X.columns if c.startswith("acc_lin_")],
    "acc":     [c for c in X.columns if c.startswith("acc_") and not c.startswith("acc_lin_")],
    "gyro":    [c for c in X.columns if c.startswith("gyro_")],
    "orient":  [c for c in X.columns if c.startswith(("yaw_", "pitch_", "roll_", "orient_"))],
    "hr":      [c for c in X.columns if c.startswith("hr_")]
}

pca_df = pd.DataFrame(index=X.index)

for group_name, cols in groups.items():
    X_group = StandardScaler().fit_transform(X[cols])

    n_comp = 3 if group_name == "hr" else 6

    pca = PCA(n_components=n_comp)
    pcs = pca.fit_transform(X_group)

    for i in range(n_comp):
        pca_df[f"{group_name}_pc{i+1}"] = pcs[:, i]

    print(
        group_name,
        "explained variance:",
        round(pca.explained_variance_ratio_.sum(), 3)
    )

acc_lin explained variance: 0.857
acc explained variance: 0.895
gyro explained variance: 0.856
orient explained variance: 0.905
hr explained variance: 0.982


In [74]:
from itertools import product

In [76]:
results = []
all_predictions = []

data = svm_df.copy()
n_repeats = 100
base_rng = np.random.default_rng(42)
meta_cols = ["set_id", "subject", "exercise", "set_nr", "focus"]
target_col = "focus"

split_groups = []

for subject in data["subject"].unique():
    for focus in data["focus"].unique():
        candidates = data[
            (data["subject"] == subject) &
            (data["focus"] == focus)
        ].index.to_numpy()

        split_groups.append(candidates)

all_test_splits = [
    np.array(split)
    for split in product(*split_groups)
]
base_rng.shuffle(all_test_splits)

n_repeats = min(9999, len(all_test_splits))
selected_test_splits = all_test_splits[:n_repeats]

print("Total possible unique splits:", len(all_test_splits))
print("Using splits:", n_repeats)

for repeat in range(n_repeats):
    if repeat % 50 == 0:
        print(f"progress: {repeat/n_repeats*100}%")

    test_idx = selected_test_splits[repeat]
    train_idx = data.index.difference(test_idx).to_numpy()

    train_df_raw = data.loc[train_idx].copy()
    test_df_raw = data.loc[test_idx].copy()

    feature_cols = [c for c in data.columns if c not in meta_cols]

    X_train_raw = train_df_raw[feature_cols].copy()
    X_test_raw = test_df_raw[feature_cols].copy()

    y_train = train_df_raw[target_col]
    y_test = test_df_raw[target_col]

    groups = {
        "acc_lin": [c for c in feature_cols if c.startswith("acc_lin_")],
        "acc": [c for c in feature_cols if c.startswith("acc_") and not c.startswith("acc_lin_")],
        "gyro": [c for c in feature_cols if c.startswith("gyro_")],
        "orient": [c for c in feature_cols if c.startswith(("yaw_", "pitch_", "roll_", "orient_"))],
        "hr": [c for c in feature_cols if c.startswith("hr_")],
    }

    train_pca_df = pd.DataFrame(index=train_df_raw.index)
    test_pca_df = pd.DataFrame(index=test_df_raw.index)

    for group_name, cols in groups.items():
        if len(cols) == 0:
            continue

        n_comp = 3 if group_name == "hr" else 6
        n_comp = min(n_comp, len(cols), len(train_df_raw) - 1)

        scaler = StandardScaler()
        pca = PCA(n_components=n_comp)

        X_train_group_scaled = scaler.fit_transform(X_train_raw[cols])
        X_test_group_scaled = scaler.transform(X_test_raw[cols])

        train_pcs = pca.fit_transform(X_train_group_scaled)
        test_pcs = pca.transform(X_test_group_scaled)

        for i in range(n_comp):
            train_pca_df[f"{group_name}_pc{i+1}"] = train_pcs[:, i]
            test_pca_df[f"{group_name}_pc{i+1}"] = test_pcs[:, i]

    train_extra = train_df_raw[["exercise", "set_nr"]].copy()
    test_extra = test_df_raw[["exercise", "set_nr"]].copy()

    train_extra = pd.get_dummies(train_extra, columns=["exercise"], prefix="exercise", dtype=int)
    test_extra = pd.get_dummies(test_extra, columns=["exercise"], prefix="exercise", dtype=int)

    test_extra = test_extra.reindex(columns=train_extra.columns, fill_value=0)

    X_train = pd.concat([train_pca_df, train_extra], axis=1)
    X_test = pd.concat([test_pca_df, test_extra], axis=1)

    svm = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=1, gamma="scale", class_weight="balanced"))
    ])

    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    results.append({
        "repeat": repeat,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "n_train": len(train_idx),
        "n_test": len(test_idx),
    })

    pred_df = test_df_raw[["set_id", "subject", "exercise", "set_nr", "focus"]].copy()
    pred_df["repeat"] = repeat
    pred_df["pred"] = y_pred
    pred_df["correct"] = pred_df["focus"] == pred_df["pred"]
    all_predictions.append(pred_df)


results_df = pd.DataFrame(results)
predictions_df = pd.concat(all_predictions, ignore_index=True)

metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1"
]

print(results_df[metric_cols].describe())

print("\nMean accuracy:", results_df["accuracy"].mean())
print("Mean balanced accuracy:", results_df["balanced_accuracy"].mean())

print("\nPrediction counts by record:")
print(
    predictions_df
    .groupby(["set_id", "subject", "exercise", "set_nr", "focus"])
    ["correct"]
    .agg(["count", "mean"])
    .sort_values("mean")
)

Total possible unique splits: 1296
Using splits: 1296
          accuracy  balanced_accuracy    precision       recall           f1
count  1296.000000        1296.000000  1296.000000  1296.000000  1296.000000
mean      0.630787           0.630787     0.639275     0.508873     0.545602
std       0.206004           0.206004     0.365324     0.301410     0.293185
min       0.250000           0.250000     0.000000     0.000000     0.000000
25%       0.500000           0.500000     0.500000     0.500000     0.500000
50%       0.625000           0.625000     0.583333     0.500000     0.666667
75%       0.750000           0.750000     1.000000     0.500000     0.666667
max       1.000000           1.000000     1.000000     1.000000     1.000000

Mean accuracy: 0.6307870370370371
Mean balanced accuracy: 0.6307870370370371

Prediction counts by record:
                                      count      mean
set_id subject exercise set_nr focus                 
22     1       2        3      1     